In [18]:
# pip install dynetx

import dynetx as dn
import networkx as nx
import pandas as pd
from dynetx import algorithms as al

n1 n2 t1
where
- n1 and n2 are nodes
- t1 is the timestamp of interaction appearance

In [24]:
g = dn.DynGraph()

with open("grafo.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        n1, n2, t = line.split()

        g.add_interaction(
            int(n1),
            int(n2),
            int(t)
        )

print("Nodi:", g.number_of_nodes())
print("Interazioni:", g.number_of_interactions())


Nodi: 37
Interazioni: 100


In [15]:
print(dir(g))
print(dir(al))

['_DynGraph__presence_test', '__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__networkx_backend__', '__networkx_cache__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_adj', '_node', 'add_cycle', 'add_edge', 'add_edges_from', 'add_interaction', 'add_interactions_from', 'add_node', 'add_nodes_from', 'add_path', 'add_star', 'add_weighted_edges_from', 'adj', 'adjacency', 'adjlist_inner_dict_factory', 'adjlist_outer_dict_factory', 'avg_number_of_nodes', 'avg_temporal_degree', 'clear', 'clear_edges', 'copy', 'coverage', 'degree', 'degree_iter', 'density', 'directed', 'edge_attr_dict_factory', 'edge_contribution', 'edge_removal', 'edge_subgraph', 'edges', 'edges_iter', 'get_edge_data', 'get

Qui sotto provo un po' delle 'analisi di base' per descrivere il grafo

In [ ]:
times = sorted(g.temporal_snapshots_ids())
times

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [26]:
rows = []

for t in times:
    # snapshot al tempo t (intervallo [t, t])
    Gt = g.time_slice(t, t)

    num_nodes = Gt.number_of_nodes()
    num_edges = Gt.number_of_edges()

    density = nx.density(Gt) if num_nodes > 1 else 0

    avg_degree = (
        sum(dict(Gt.degree()).values()) / num_nodes
        if num_nodes > 0 else 0
    )

    if num_nodes > 0:
        num_components = nx.number_connected_components(Gt)
        largest_cc = max(len(c) for c in nx.connected_components(Gt))
    else:
        num_components = 0
        largest_cc = 0

    clustering = (
        nx.average_clustering(Gt)
        if num_edges > 0 else 0
    )

    rows.append({
        "time": t,
        "nodes": num_nodes,
        "edges": num_edges,
        "density": density,
        "avg_degree": avg_degree,
        "components": num_components,
        "largest_cc": largest_cc,
        "clustering": clustering
    })

df = pd.DataFrame(rows).sort_values("time")
print(df)


    time  nodes  edges   density  avg_degree  components  largest_cc  \
0      0     12     12  0.181818    2.000000           4           3   
1      1     15      9  0.085714    1.200000           7           3   
2      2     14     11  0.120879    1.571429           3           6   
3      3     18      9  0.058824    1.000000           9           2   
4      4     10     11  0.244444    2.200000           3           4   
5      5      8      8  0.285714    2.000000           1           8   
6      6     18      9  0.058824    1.000000           9           2   
7      7     18      9  0.058824    1.000000           9           2   
8      8      9      9  0.250000    2.000000           3           3   
9      9     12      6  0.090909    1.000000           6           2   
10    10      7      7  0.333333    2.000000           1           7   
11    11     12      6  0.090909    1.000000           6           2   
12    12     11      6  0.109091    1.090909           5        

ora provo: per ogni nodo v e per ogni tempo t, il numero di vicini di v nello snapshot al tempo t
ci serve per le rq1

In [ ]:

times = sorted(g.temporal_snapshots_ids())
nodes = sorted(g.nodes())

# grado per ogni nodo
rows = []

for t in times:
    Gt = g.time_slice(t, t)  # snapshot al tempo t

    for v in nodes:
        if v in Gt:
            degree = Gt.degree(v)
        else:
            degree = 0

        rows.append({
            "time": t,
            "node": v,
            "degree": degree
        })



df = pd.DataFrame(rows)

print(df.head(5))

   time  node  degree
0     0     0       2
1     0     1       2
2     0     2       2
3     0     3       2
4     0     4       2


ora vedo come fare analisi delle comunità (ci servono per la rq2)
provo louvain

In [ ]:
# pip install python-louvain


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
